# Notebook 01: Bienvenida y Tu Primer Data Doc

## Taller de Great Expectations!

En los próximos módulos aprenderás a:
- Validar la calidad de tus datos
- Documentar reglas de negocio
- Generar reportes profesionales
- Automatizar validaciones en producción


## ¿Qué es Calidad de Datos?

La **calidad de datos** es el grado en que los datos cumplen con los requisitos de uso previstos.

### ¿Por qué importa?

- **Decisiones incorrectas**: Datos malos → Decisiones malas
- **Costos**: Se estima que cuesta 15-25% de los ingresos
- **Riesgos**: Incumplimiento regulatorio, pérdida de confianza

### Ejemplo Real

Imagina un e-commerce donde:
-  Precios negativos → Pérdidas financieras
-  IDs de cliente nulos → No puedes contactar al cliente
-  Fechas futuras → Reportes incorrectos

## ¿Qué es Great Expectations?

**Great Expectations (GX)** es una herramienta open-source para:

1. **Validar** datos contra reglas definidas
2. **Documentar** esas reglas automáticamente
3. **Generar reportes** visuales (Data Docs)
4. **Automatizar** validaciones en pipelines

### Filosofía

> "Las expectativas sobre tus datos deben ser explícitas, versionadas y testeables"

In [1]:
# Importar librerías
import great_expectations as gx
import pandas as pd

print(f"Great Expectations versión: {gx.__version__}")

Great Expectations versión: 1.13.0


## Tu Primera Validación

Vamos a validar un dataset de ventas con problemas de calidad.

In [2]:
# Cargar datos
df = pd.read_csv("../data/ventas_sucias.csv")

print(f"Total de registros: {len(df)}")
print(f"\nPrimeras filas:")
df.head()

Total de registros: 1500

Primeras filas:


,order_id,order_date,customer_id,product_category,price,quantity
0,d3ebe033-74f3-4ad9-8ae3-1c2a817a8811,NaN,5506.0,Electronics,48.03,4
1,25ca530a-48a1-4f9f-9cbe-1ab48a5a7cd2,2023-10-15,4257.0,Toys,405.67,-4
2,bf95eb3f-a26c-4d63-903b-9b04d3567b77,2023-03-21,4527.0,Electronics,175.26,8
3,fe4adf9b-b8d6-4a70-998e-75570bf76502,2023-07-13,2291.0,Home,27.68,2
4,f8412298-6cec-4526-bd07-1fcc7f1543f3,2023-07-14,5554.0,Clothing,137.15,10


In [3]:
# Análisis rápido de problemas
print("=== PROBLEMAS DETECTADOS ===")
print(f"\nValores nulos:")
print(df.isnull().sum())

print(f"\nPrecios negativos: {(df['price'] < 0).sum()}")
print(f"Cantidades <= 0: {(df['quantity'] <= 0).sum()}")

=== PROBLEMAS DETECTADOS ===

Valores nulos:
order_id             0
order_date          76
customer_id         25
product_category    27
price                0
quantity             0
dtype: int64

Precios negativos: 49
Cantidades <= 0: 69


## Crear Tu Primera Expectativa

Una **Expectation** es una regla sobre tus datos. Por ejemplo:
- "Los precios deben ser positivos"
- "El customer_id no debe ser nulo"

## ¿Qué es el "Context" en Great Expectations?
El Data Context es el "cerebro" de tu proyecto. Es el objeto que gestiona todas las configuraciones, como:

Fuentes de datos: De dónde vienen tus datos (Pandas, Spark, SQL).

Expectations: Las reglas que deben cumplir tus datos.

Checkpoints: Cuándo y cómo se ejecutan las validaciones.

Data Docs: La visualización de los resultados.

## ¿Qué significa mode="ephemeral"?
Cuando usas el modo efímero, estás creando un contexto en memoria. Esto tiene implicaciones importantes:

Sin archivos locales: No se crea la carpeta great_expectations/ ni archivos YAML en tu disco duro.

Temporalidad: Todo lo que configures (Data Sources, Expectation Suites) vive solo mientras tu script de Python esté en ejecución. Una vez que el proceso termina, la configuración desaparece.

Uso ideal: Es perfecto para entornos de Notebooks (Jupyter/Colab), pruebas rápidas, o procesos de CI/CD donde no quieres arrastrar archivos de configuración persistentes.

In [4]:
# Inicializar contexto
context = gx.get_context(mode="ephemeral")

# Configurar datasource
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

print(" Contexto configurado")

 Contexto configurado


In [5]:
# Crear suite de expectativas
suite = context.suites.add(gx.ExpectationSuite(name="mi_primera_suite"))

# Agregar expectativas simples
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id")
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price",
        min_value=0.01
    )
)

suite.save()
print(f" Suite creada con {len(suite.expectations)} expectativas")

 Suite creada con 2 expectativas


## Ejecutar la Validación

In [6]:
# Crear validation definition
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        data=batch_def,
        suite=suite,
        name="primera_validacion"
    )
)

# Ejecutar validación
resultado = validation_def.run(batch_parameters={"dataframe": df})

print("\n" + "="*50)
print("RESULTADO DE LA VALIDACIÓN")
print("="*50)
print(f"\n¿Validación exitosa?: {resultado.success}")
print(f"Expectativas evaluadas: {len(resultado.results)}")
print(f"Expectativas que pasaron: {sum(1 for r in resultado.results if r.success)}")
print(f"Expectativas que fallaron: {sum(1 for r in resultado.results if not r.success)}")

Calculating Metrics:   0%|          | 0/15 [00:00<?, ?it/s]


RESULTADO DE LA VALIDACIÓN

¿Validación exitosa?: False
Expectativas evaluadas: 2
Expectativas que pasaron: 0
Expectativas que fallaron: 2


## Tu Primer Data Doc

Ahora viene la magia: Great Expectations genera automáticamente documentación HTML interactiva.

In [7]:
# Generar Data Docs
context.build_data_docs()

print("\n" + "="*50)
print(" Data Docs generados exitosamente!")
print("="*50)
print("\nAbriendo en tu navegador...")

# Abrir en navegador
context.open_data_docs()


 Data Docs generados exitosamente!

Abriendo en tu navegador...


## ¿Qué Ves en el Data Doc?

En la página que se abrió, observa:

1. **Overview**: Resumen de la validación
2. **Expectation Suite**: Todas tus reglas documentadas
3. **Validation Results**: Resultados detallados
4. **Statistics**: Gráficos y métricas

### Ventajas

-  **Visual**: Fácil de entender para no-técnicos
-  **Compartible**: Puedes enviar el link
-  **Automático**: Se genera sin esfuerzo
-  **Profesional**: Listo para stakeholders